#Data Splitting

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load cleaned dataset
df = pd.read_csv("clean_reviews.csv")

# First split: 70% train, 30% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["Score"],
    random_state=42
)

# Second split: temp -> 10% validation, 20% test
# Since temp is 30%, splitting 1/3 and 2/3 gives 10% and 20% of full data
val_df, test_df = train_test_split(
    temp_df,
    test_size=2/3,
    stratify=temp_df["Score"],
    random_state=42
)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (313371, 4)
Validation shape: (44767, 4)
Test shape: (89535, 4)


#Check The Split Ratios

In [2]:
total = len(df)

print("Train %:", len(train_df) / total)
print("Validation %:", len(val_df) / total)
print("Test %:", len(test_df) / total)

Train %: 0.6999997766226688
Validation %: 0.09999932986800633
Test %: 0.20000089350932487


#Check class distributions

In [3]:
print("Train distribution:")
print(train_df["Score"].value_counts(normalize=True).sort_index())

print("\nValidation distribution:")
print(val_df["Score"].value_counts(normalize=True).sort_index())

print("\nTest distribution:")
print(test_df["Score"].value_counts(normalize=True).sort_index())

Train distribution:
Score
1    0.098720
2    0.054983
3    0.077879
4    0.141624
5    0.626794
Name: proportion, dtype: float64

Validation distribution:
Score
1    0.098733
2    0.054974
3    0.077892
4    0.141622
5    0.626779
Name: proportion, dtype: float64

Test distribution:
Score
1    0.098721
2    0.054984
3    0.077880
4    0.141621
5    0.626794
Name: proportion, dtype: float64


#Resampling
---
Resample Training set Only

#Check If 15000 Sample Per Class Is Feasible

In [4]:
print(train_df["Score"].value_counts().sort_index())

Score
1     30936
2     17230
3     24405
4     44381
5    196419
Name: count, dtype: int64


#Take 15000 Sample Per Class

In [5]:
train_balanced_df = (
    train_df.groupby("Score", group_keys=False)
    .apply(lambda x: x.sample(n=15000, random_state=42, replace=True), include_groups=True)
    .reset_index(drop=True)
)

/tmp/ipykernel_3425/1671553545.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=15000, random_state=42, replace=True), include_groups=True)


In [6]:
#confirm the sampling was successful
print(train_balanced_df["Score"].value_counts().sort_index())

Score
1    15000
2    15000
3    15000
4    15000
5    15000
Name: count, dtype: int64


#Saving Files




In [ ]:
train_balanced_df.to_csv("train_balanced.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

The cleaned dataset was split into training, validation, and test sets using stratified sampling with a ratio of 70:10:20 to preserve the original class distribution across all subsets. To address class imbalance during model training, only the training set was balanced by sampling 15,000 reviews from each score category. The validation and test sets were kept in their natural distributions to provide a more realistic evaluation of model performance.

#Feature Engineering

In [14]:
import pandas as pd

train_df = pd.read_csv("train_balanced.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

print(train_df.shape, val_df.shape, test_df.shape)

(75000, 4) (44767, 4) (89535, 4)


#Prepare Inputs And Labels

In [15]:
X_train_text = train_df["review_text"]
X_val_text = val_df["review_text"]
X_test_text = test_df["review_text"]

y_train = train_df["Score"]
y_val = val_df["Score"]
y_test = test_df["Score"]

#Feature Set 1 — TF-IDF + N-grams

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=20000,
    min_df=5
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)
X_test_tfidf = tfidf.transform(X_test_text)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF validation shape:", X_val_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)

TF-IDF train shape: (75000, 20000)
TF-IDF validation shape: (44767, 20000)
TF-IDF test shape: (89535, 20000)


#Feature Set 2 — Average Word2Vec

In [17]:
#check tokens
print(train_df["tokens"].iloc[0])
print(type(train_df["tokens"].iloc[0]))

['spoiled', 'returnable', 'put', 'son', 'christmas', 'stocking', 'opened', 'oil', 'separated', 'orange', 'disguisting', 'messi', 'order', 'pack', 'repalcement', 'gift', 'arrived', 'thing', 'happened', 'two', 'jar', 'tried', 'return', 'find', 'nonreturnablei', 'emailed', 'never', 'received', 'replyif', 'want', 'baconnaise', 'suggest', 'try', 'find', 'locally', 'see', 'still', 'good', 'buy']
<class 'str'>


In [18]:
import ast

df["tokens"] = df["tokens"].apply(ast.literal_eval)

In [19]:
print(df["tokens"].iloc[0])
print(type(df["tokens"].iloc[0]))

['good', 'quality', 'dog', 'food', 'bought', 'several', 'vitality', 'canned', 'dog', 'food', 'product', 'found', 'good', 'quality', 'product', 'look', 'like', 'stew', 'processed', 'meat', 'smell', 'better', 'labrador', 'finicky', 'appreciates', 'product', 'better']
<class 'list'>


#Train Word2Vec On Training Tokens Only

In [20]:
!pip install gensim
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=train_df["tokens"],
    vector_size=100,
    window=5,
    min_count=5,
    workers=4
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 21.0 MB/s eta 0:00:00


#Create Average Review Vectors

In [21]:
import numpy as np

def average_word2vec(tokens, model, vector_size):
    valid_vectors = [
        model.wv[word] for word in tokens
        if word in model.wv
    ]

    if len(valid_vectors) == 0:
        return np.zeros(vector_size)

    return np.mean(valid_vectors, axis=0)

In [22]:
X_train_w2v = np.array([
    average_word2vec(tokens, w2v_model, 100)
    for tokens in train_df["tokens"]
])

X_val_w2v = np.array([
    average_word2vec(tokens, w2v_model, 100)
    for tokens in val_df["tokens"]
])

X_test_w2v = np.array([
    average_word2vec(tokens, w2v_model, 100)
    for tokens in test_df["tokens"]
])

print("Word2Vec train shape:", X_train_w2v.shape)
print("Word2Vec validation shape:", X_val_w2v.shape)
print("Word2Vec test shape:", X_test_w2v.shape)

Word2Vec train shape: (75000, 100)
Word2Vec validation shape: (44767, 100)
Word2Vec test shape: (89535, 100)


#Save TF-IDF Matrices

In [24]:
from scipy.sparse import save_npz

save_npz("X_train_tfidf.npz", X_train_tfidf)
save_npz("X_val_tfidf.npz", X_val_tfidf)
save_npz("X_test_tfidf.npz", X_test_tfidf)

#Save Word2Vec Arrays

In [25]:
np.save("X_train_w2v.npy", X_train_w2v)
np.save("X_val_w2v.npy", X_val_w2v)
np.save("/X_test_w2v.npy", X_test_w2v)

#Save Labels


In [26]:
np.save("/y_train.npy", y_train.to_numpy())
np.save("y_val.npy", y_val.to_numpy())
np.save("y_test.npy", y_test.to_numpy())

Two feature representations were constructed for the review text. First, TF-IDF with unigram and bigram features was used with a maximum vocabulary size of 20,000 and a minimum document frequency of 5. The vectorizer was fitted only on the balanced training set, and the learned vocabulary was then used to transform the validation and test sets. Second, Average Word2Vec embeddings were generated by training a Word2Vec model on the tokenized training reviews only. Each review was represented as the average of its word vectors, resulting in a fixed-length 100-dimensional feature vector.

In [ ]:
import json
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

with open(filename, 'r') as f:
    nb = json.load(f)

# Completely remove widgets metadata (safest for GitHub)
if 'metadata' in nb:
    if 'widgets' in nb['metadata']:
        del nb['metadata']['widgets']

# Save and download
with open(f"feature_engineering_{filename}", 'w') as f:
    json.dump(nb, f, indent=2)

files.download(f"feature_engineering_{filename}")

Saving amazon_fine_food_review_score_prediction_train&test_split.ipynb to amazon_fine_food_review_score_prediction_train&test_split.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>